# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. 
2. 
3. 
4. 
5.

In [10]:
import numpy as np
# 1
# a = np.arange(6).reshape(2, 3)
# print(a[1, 2], a.shape)
# 2
# a = np.array([3, 8, 1, 8])
# print(a.argmax(), a.max())
# # 3
# a = np.arange(10);
# print(a)
# b = a[::3];
# print(b)
# b[0] = 99; print(a[0])
# # 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
print((a > 3))
# # 5
# print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])

2 0.5


In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [21]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
##Дополнительная функция зачистка от дубль платежей
seen = set()
unique_rows = []

for r in rows:
    row_tuple = tuple(r.values())
    if row_tuple not in seen:
        seen.add(row_tuple)
        unique_rows.append(r)
rows = unique_rows
##Дополнительная функция зачистка от дубль платежей

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5283,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [36]:
# База
#1 Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
valid_mask = paid & (cust != "TEST")
filtered_amounts = amount[valid_mask]
max_idx = np.argmax(filtered_amounts)

filtered_cust = cust[valid_mask]
filtered_city = city[valid_mask]

print("Самый дорогой оплаченный не тестовый заказ:", filtered_amounts[max_idx])
print("ID клиента:", filtered_cust[max_idx])
print("Город:", filtered_city[max_idx])

Самый дорогой оплаченный не тестовый заказ: 139040.0
ID клиента: C02122
Город: Шымкент


In [37]:
#2 Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
valid_mask = paid & (cust != "TEST")
filtered_amounts = amount[valid_mask]
summ = np.sum(filtered_amounts)
print(summ)
valid_mask = ~paid & (cust != "TEST")
filtered_amounts = amount[valid_mask]
summ = np.sum(filtered_amounts)
print(summ)


69662930.0
7778880.0


In [34]:
#3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?
valid_mask = paid & (cust != "TEST")
valid_mask_shymkent = valid_mask& (city == "Шымкент")

print(f"Общее количество заказов {amount[valid_mask].shape[0]:<10} / {amount[valid_mask].sum():<10,} из Шымкента {amount[valid_mask_shymkent].shape[0]:>7} / {amount[valid_mask_shymkent].sum():<10,}")

count_shymkent = amount[valid_mask_shymkent].shape[0]
count_total = amount[valid_mask].shape[0]

share_percent = (count_shymkent / count_total) * 100
share_percent = np.mean(city[valid_mask] == "Шымкент") * 100
print(f"Доля заказов из Шымкента: {share_percent:.2f}%")

Общее количество заказов 4739       / 69,662,930.0 из Шымкента     616 / 8,602,660.0
Доля заказов из Шымкента: 13.00%


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [27]:
# Стандарт
# 4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
#Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
discount=15
discount_price=5000
valid_mask = paid
valid_mask_discount = valid_mask& (amount > discount_price)
valid_mask_discount_without_test = valid_mask_discount & (cust != "TEST")
print(cust[valid_mask_discount].shape[0])
print(cust[valid_mask_discount_without_test].shape[0])

# discount_np_amount=valid_mask_discount & amount[amount>discount_price]
discount_rate = discount / 100
all_discounts = np.where(valid_mask_discount, amount * discount_rate, 0.0)
all_discounts_without_test = np.where(valid_mask_discount_without_test, amount * discount_rate, 0.0)

print(f"кол-во заказов {amount[valid_mask_discount].shape[0]:<7} / {amount[valid_mask_discount].sum():<10,}  / {all_discounts[valid_mask_discount].sum():<10,} ")
print(f"кол-во заказов {amount[valid_mask_discount_without_test].shape[0]:<7} / {amount[valid_mask_discount_without_test].sum():<10,}  / {all_discounts_without_test[valid_mask_discount].sum():<10,}")

3834
3831
кол-во заказов 3834    / 96,508,210.0  / 14,476,231.5 
кол-во заказов 3831    / 66,508,240.0  / 9,976,236.0


In [49]:
# 5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
# days
cityList=np.unique(city)

valid_mask = paid & ((days == 1) | (days == 2)) & (cust != "TEST")
for cityName in cityList:
    valid_mask_city = valid_mask & (city == cityName)
    # count = amount[valid_mask_city].shape[0]
    # count_total = amount[valid_mask].shape[0]
    # share_percent = (count / count_total) * 100
    share_percent = np.mean(city[valid_mask] == cityName) * 100
    print(f"Доля заказов из {cityName:<7}: {share_percent:.2f}%")

Доля заказов из Актобе : 7.46%
Доля заказов из Алматы : 43.87%
Доля заказов из Астана : 27.42%
Доля заказов из Атырау : 3.00%
Доля заказов из Караганда: 11.07%
Доля заказов из Шымкент: 7.19%


In [51]:
# 6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
valid_mask = paid & (cust != "TEST")

# avg = amount.mean()
# Не применены фильтры
avg = amount[valid_mask].mean()
print("Средний чек:", round(avg))

Средний чек: 14700


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [53]:
# Челлендж
#7
import time
import numpy as np

# --- 1. Замер через цикл for ---
start_time = time.perf_counter()  # Запускаем секундомер

total_sum_for = 0
for i in range(len(amount)):
    if paid[i] and cust[i] != "TEST":
        total_sum_for += amount[i]

end_time = time.perf_counter()    # Останавливаем секундомер
time_for = end_time - start_time


# --- 2. Замер через NumPy-маску ---
start_time = time.perf_counter()

valid_mask = paid & (cust != "TEST")
total_sum_np = np.sum(amount[valid_mask])

end_time = time.perf_counter()
time_np = end_time - start_time


# --- 3. Считаем во сколько раз быстрее ---
speedup = time_for / time_np

# Правильно: текст «сек» пишется снаружи закрывающей скобки }
print(f"Сумма через цикл: {total_sum_for:,.2f} (Время: {time_for:.6f} сек)")
print(f"Сумма через NumPy: {total_sum_np:,.2f} (Время: {time_np:.6f} сек)")
print(f"NumPy оказался быстрее в {speedup:.1f} раз(а)!")

Сумма через цикл: 69,662,930.00 (Время: 0.002597 сек)
Сумма через NumPy: 69,662,930.00 (Время: 0.000662 сек)
NumPy оказался быстрее в 3.9 раз(а)!
